# 从算子层验证 Attention 理论

05.02 提出了五条可检验的判断。本节把所有 NPU 算子实验统一到 Qwen3-1.7B 的长序列 shape：BF16、`Nq=16, Nkv=8, D=128, S/T=4096`。不再混用 256、512、1024、2048 等教学 shape，避免读者在代码和结论之间反复换算。

§3 在同一个 `S=4096` 输入上比较 eager、SDPA 和直接 FusionAttention V3，验证 dispatch、数值一致性、wall-time、峰值显存与 kernel 数。§4 只切换 $N_{kv}$ 验证 GQA 的收益边界。§5 的六 token CPU 例子只用于解释 mask 语义，不属于性能 shape。§6 固定 `T=4096`，只改变文档划分，验证 VarLen 是否跳过跨文档 pair。

所有性能结论保留两种口径：关闭 profiler 的重复 wall-time 回答“快不快”，trace 回答“实际跑了什么”。

## 1. 环境、Shape 与测量协议

所有 NPU 实验固定 `S/T=4096, Nq=16, D=128, dtype=BF16`。基础算子和 VarLen 使用 `B=1`；GQA 为保持原有模型对照使用 `B=2`，只改变 `Nkv=16/8`。

唯一允许变化的量是当前实验要验证的变量：算子实现、K/V head 数或 `cu_seq` 文档划分。六 token mask 例子在 CPU 上运行，只验证语义。

计时：5 次 warmup、20 次重复，区间前后 `torch.npu.synchronize()`。Profiler 只用于归因——不拿 profiler 内部 duration 替代稳定 wall-time。

In [ ]:
import gc
import json
import platform
import time
from collections import Counter
from pathlib import Path

import torch
import torch.nn.functional as F
import torch_npu

assert torch.npu.is_available(), "本节需要可用的 Ascend NPU 环境"
torch.npu.set_device(0)

DEVICE = torch.device("npu:0")
DTYPE = torch.bfloat16
SEED = 2026
NQ, NKV, D = 16, 8, 128
SCALE = D ** -0.5

SEQ_LEN = 4096
BASE_BATCH, GQA_BATCH = 1, 2
WARMUP, REPEATS = 5, 20

COMPRESSED_CAUSAL_MASK = torch.triu(
    torch.ones(2048, 2048, dtype=torch.bool, device=DEVICE), diagonal=1
)
EAGER_CAUSAL_MASK = torch.triu(
    torch.ones(SEQ_LEN, SEQ_LEN, dtype=torch.bool, device=DEVICE), diagonal=1
)


def seed_all(offset=0):
    torch.manual_seed(SEED + offset)


def make_inputs(*, batch_size=BASE_BATCH, n_kv=NKV, requires_grad=False, seed_offset=0):
    seed_all(seed_offset + batch_size + n_kv)
    q = torch.randn(batch_size, SEQ_LEN, NQ, D, device=DEVICE, dtype=DTYPE)
    k = torch.randn(batch_size, SEQ_LEN, n_kv, D, device=DEVICE, dtype=DTYPE)
    v = torch.randn(batch_size, SEQ_LEN, n_kv, D, device=DEVICE, dtype=DTYPE)
    return tuple(x.requires_grad_(requires_grad) for x in (q, k, v))


def mib(num_bytes):
    return num_bytes / 1024**2


print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("torch_npu:", torch_npu.__version__)
print("NPU:", torch.npu.get_device_name(0))
print("warmup / repeats:", WARMUP, "/", REPEATS)


## 2. 四条算子路径

- **eager reference**：在 `S=4096` 显式扩展 GQA 的 K/V，物化 score，执行 mask、softmax 和 `PV`。
- **PyTorch SDPA**：统一 API，dispatcher 选 NPU backend。
- **FusionAttention V3 dense**：直接调用 BSND、`sparse_mode=3` 的 NPU 原生融合算子。
- **FusionAttention V3 VarLen**：BSND → TND，传累计长度，`sparse_mode=7`。

前三条在相同 causal 语义下可做数值 sanity check。VarLen 多文档路径语义不同，需与显式 block-document eager reference 对照。

In [ ]:
def pytorch_sdpa(q, k, v):
    enable_gqa = q.size(2) != k.size(2)
    out = F.scaled_dot_product_attention(
        q.transpose(1, 2),
        k.transpose(1, 2),
        v.transpose(1, 2),
        scale=SCALE,
        is_causal=True,
        enable_gqa=enable_gqa,
    )
    return out.transpose(1, 2)


def fusion_attention_v3(q, k, v):
    seq_len = q.size(1)
    return torch.ops.npu.npu_fusion_attention_v3(
        query=q,
        key=k,
        value=v,
        head_num=q.size(2),
        input_layout="BSND",
        atten_mask=COMPRESSED_CAUSAL_MASK,
        scale=SCALE,
        keep_prob=1.0,
        pre_tockens=seq_len,
        next_tockens=0,
        sparse_mode=3,
    )[0]


def eager_attention(q, k, v, *, allowed_mask=None):
    q_bnsd = q.transpose(1, 2)
    k_bnsd = k.transpose(1, 2)
    v_bnsd = v.transpose(1, 2)

    if q_bnsd.size(1) != k_bnsd.size(1):
        repeat = q_bnsd.size(1) // k_bnsd.size(1)
        k_bnsd = k_bnsd.repeat_interleave(repeat, dim=1)
        v_bnsd = v_bnsd.repeat_interleave(repeat, dim=1)

    scores = torch.matmul(q_bnsd, k_bnsd.transpose(-2, -1)) * SCALE
    if allowed_mask is None:
        scores = scores.masked_fill(EAGER_CAUSAL_MASK, float("-inf"))
    else:
        scores = scores.masked_fill(~allowed_mask, float("-inf"))
    probs = torch.softmax(scores, dim=-1)
    return torch.matmul(probs, v_bnsd).transpose(1, 2)


def make_cu_seq(lengths):
    lengths = torch.as_tensor(lengths, dtype=torch.int64, device="cpu")
    if lengths.ndim != 1 or (lengths <= 0).any():
        raise ValueError("lengths 必须是一维正整数")
    return torch.cat([torch.zeros(1, dtype=torch.int64), lengths.cumsum(0)])


def varlen_attention_v3(q, k, v, cu_seq):
    batch_size, seq_len, n_q, head_dim = q.shape
    if batch_size != 1:
        raise ValueError("本实验的 TND sweep 固定 batch_size=1")
    if seq_len != SEQ_LEN or int(cu_seq[-1]) != SEQ_LEN:
        raise ValueError(f"本实验固定 T={SEQ_LEN}，且 cu_seq[-1] 必须等于 T")

    q_tnd = q.reshape(seq_len, n_q, head_dim)
    k_tnd = k.reshape(seq_len, k.size(2), head_dim)
    v_tnd = v.reshape(seq_len, v.size(2), head_dim)
    actual_seq = cu_seq.to(dtype=torch.int64, device="cpu")[1:]

    out_tnd = torch.ops.npu.npu_fusion_attention_v3(
        query=q_tnd,
        key=k_tnd,
        value=v_tnd,
        head_num=n_q,
        input_layout="TND",
        atten_mask=COMPRESSED_CAUSAL_MASK,
        scale=SCALE,
        keep_prob=1.0,
        pre_tockens=seq_len,
        next_tockens=0,
        actual_seq_qlen=actual_seq,
        actual_seq_kvlen=actual_seq,
        sparse_mode=7,
    )[0]
    return out_tnd.view(batch_size, seq_len, n_q, head_dim)


def document_allowed_mask(lengths, *, device):
    doc_ids = torch.repeat_interleave(
        torch.arange(len(lengths), device=device),
        torch.as_tensor(lengths, device=device),
    )
    same_document = doc_ids[:, None] == doc_ids[None, :]
    causal = torch.arange(doc_ids.numel(), device=device)[:, None] >= torch.arange(
        doc_ids.numel(), device=device
    )[None, :]
    return same_document & causal


### 统一的计时与显存口径

计时函数每次重新构造输入，forward 与 fwd+bwd 分开测。峰值显存测量的是单次 forward 新增的 active allocation——只反映 score、概率和临时 workspace，不把 Q/K/V 输入大小重复计入。

In [ ]:
def benchmark_case(
    fn,
    *,
    batch_size=BASE_BATCH,
    n_kv=NKV,
    backward=False,
    warmup=WARMUP,
    repeats=REPEATS,
):
    q, k, v = make_inputs(
        batch_size=batch_size,
        n_kv=n_kv,
        requires_grad=backward,
    )
    grad = torch.randn_like(q) if backward else None

    def step():
        if backward:
            out = fn(q, k, v)
            out.backward(grad)
            q.grad = k.grad = v.grad = None
        else:
            with torch.no_grad():
                fn(q, k, v)

    for _ in range(warmup):
        step()
    torch.npu.synchronize()
    start = time.perf_counter()
    for _ in range(repeats):
        step()
    torch.npu.synchronize()
    elapsed_ms = (time.perf_counter() - start) * 1000 / repeats

    del q, k, v, grad
    gc.collect()
    return elapsed_ms


def forward_peak_mib(fn, *, batch_size=BASE_BATCH, n_kv=NKV):
    q, k, v = make_inputs(batch_size=batch_size, n_kv=n_kv)
    with torch.no_grad():
        fn(q, k, v)  # 建好 lazy workspace/cache 后再测
    torch.npu.synchronize()

    baseline = torch.npu.memory_allocated()
    torch.npu.reset_peak_memory_stats()
    with torch.no_grad():
        out = fn(q, k, v)
    torch.npu.synchronize()
    peak_delta = torch.npu.max_memory_allocated() - baseline

    del out, q, k, v
    gc.collect()
    return mib(max(0, peak_delta))


def print_rows(headers, rows):
    widths = [max(len(str(header)), *(len(str(row[i])) for row in rows)) for i, header in enumerate(headers)]
    print("  ".join(str(header).rjust(widths[i]) for i, header in enumerate(headers)))
    print("  ".join("-" * width for width in widths))
    for row in rows:
        print("  ".join(str(value).rjust(widths[i]) for i, value in enumerate(row)))


## 3. SDPA Dispatch 与 FlashAttention 融合收益

三条 causal 路径都使用 `B=1,S=4096`：先做 BF16 数值 sanity check，再测 forward、fwd+bwd 和单次 forward 新增峰值显存。没有额外的短序列预检或显存 sweep。

理论成立的证据应当是：

1. SDPA trace 实际出现 FlashAttentionScore，而不是仅凭 API 名称猜测；
2. eager 显式物化 `[1,16,4096,4096]` score，单个 BF16 score 的理论大小就是 `512 MiB`；
3. SDPA/V3 的峰值不包含完整 score，且 trace 中的 device kernel 数显著少于 eager。单一 shape 只能证明 `S=4096` 这一点，不再外推显存随 S 的增长曲线。

In [ ]:
check_q, check_k, check_v = make_inputs()
with torch.no_grad():
    eager_out = eager_attention(check_q, check_k, check_v)
    sdpa_out = pytorch_sdpa(check_q, check_k, check_v)
    v3_out = fusion_attention_v3(check_q, check_k, check_v)

print("causal sanity check (BF16):")
print("  eager vs SDPA max / mean abs:",
      float((eager_out - sdpa_out).abs().max()),
      float((eager_out - sdpa_out).abs().mean()))
print("  eager vs V3   max / mean abs:",
      float((eager_out - v3_out).abs().max()),
      float((eager_out - v3_out).abs().mean()))
del check_q, check_k, check_v, eager_out, sdpa_out, v3_out

fusion_rows = []
for name, fn in {
    "eager": eager_attention,
    "pytorch_sdpa": pytorch_sdpa,
    "fusion_attention_v3": fusion_attention_v3,
}.items():
    fusion_rows.append((
        name,
        f"{benchmark_case(fn):.3f}",
        f"{benchmark_case(fn, backward=True):.3f}",
        f"{forward_peak_mib(fn):.1f}",
    ))

print(f"\nB={BASE_BATCH}, S={SEQ_LEN} fixed-shape validation:")
print_rows(("path", "forward ms", "fwd+bwd ms", "forward peak MiB"), fusion_rows)


### 小结

三条路径只比较 `B=1,S=4096`。先看数值误差，确认 eager、SDPA 和直接 V3 仍实现同一个 causal attention；再看单一 shape 的 wall-time 与峰值显存。

eager 的逻辑 score 形状是 `[1,16,4096,4096]`，单个 BF16 score 就占 `512 MiB`，而 eager 还要继续生成 softmax 概率。SDPA/V3 通过 tiling 和 online softmax 避免物化这个完整中间张量。因此本节能给出的严谨结论是：**在 `S=4096` 上，融合路径避免了完整 score 成本并减少算子碎片。**由于已经删除多 S sweep，这里不再声称观察到了渐近增长曲线，也不推断收益从哪个序列长度开始出现。

§7 再用同为 `S=4096` 的 trace 确认 SDPA 是否实际命中 FlashAttentionScore，以及 dispatch 路径比直接 V3 多出哪些 kernel。

## 4. Qwen3 GQA 的收益边界

保持 `B=2,S=4096,Nq=16,D=128` 不变，只比较 `Nkv=16`（MHA）与 `Nkv=8`（Qwen3 GQA）。

直接读取实际 K/V tensor 的 `numel × element_size`，同时报告逻辑 score 大小。预期：K/V 字节数减半，score 由 `Nq=16` 决定不变。SDPA wall-time 未必减半——QK/PV 主计算和输出 head 数没有变化，backend 还可能在内部扩展 K/V。

In [ ]:
gqa_rows = []
logical_score_mib = mib(GQA_BATCH * NQ * SEQ_LEN**2 * torch.tensor([], dtype=DTYPE).element_size())

for label, n_kv in (("MHA", NQ), ("Qwen3 GQA", NKV)):
    q, k, v = make_inputs(batch_size=GQA_BATCH, n_kv=n_kv)
    kv_mib = mib((k.numel() + v.numel()) * k.element_size())
    del q, k, v
    gqa_rows.append((
        label,
        n_kv,
        f"{kv_mib:.1f}",
        f"{logical_score_mib:.1f}",
        f"{benchmark_case(pytorch_sdpa, batch_size=GQA_BATCH, n_kv=n_kv):.3f}",
        f"{benchmark_case(pytorch_sdpa, batch_size=GQA_BATCH, n_kv=n_kv, backward=True):.3f}",
    ))

print_rows(
    ("config", "Nkv", "K+V MiB", "logical score MiB", "forward ms", "fwd+bwd ms"),
    gqa_rows,
)


### 小结

`Nkv=16→8` 让 K/V tensor 从 `64.0 MiB` 精确减半到 `32.0 MiB`，而逻辑 score 保持 `1024.0 MiB` 不变——这是由 tensor shape 直接得到的结果：score 由 query head 数 $N_q=16$ 和序列长度决定，跟 K/V head 数无关。

从本组 `B=2,S=4096,Nq=16` 的测量可以观察到另一条趋势：forward 从 `1.581 ms` 降到 `1.482 ms`，快 6.3%；fwd+bwd 快 8.7%，都远小于 K/V 字节数的 50% 降幅。这与 $QK^\top$、$PV$ 的逻辑工作仍由 $N_q$ 驱动相符，也说明当前 kernel 的总耗时并不只由 K/V tensor 字节数决定。仅凭这两个测量点不能断言其他 shape 或 backend 也会得到 6.3%/8.7%；它们支持的结论是：本节 workload 中，K/V 减半没有带来整个 SDPA 算子的近 2× 加速。

因此要把两种结论分开：**GQA 缩小 K/V projection、activation 和 KV cache 是结构上的确定变化；SDPA wall-time 改善多少则是需要逐 workload 测量的实现结果。**

## 5. SFT Mask 语义验证

把 Q/K 全设为 0，让 softmax 在允许位置上均匀分配权重。Value 依次为 `[1,2,3,10,20,30]`，前 3 个 token 属文档 0，后 3 个属文档 1。

对文档 1 的第一个 token（位置 3）：

- 只有 causal mask：平均读取 `[1,2,3,10]`，输出应为 4；
- block-document causal mask：只能读本文件的 `[10]`，输出应为 10；
- 修改 labels/`IGNORE_INDEX`：不改变 SDPA 输出——labels 不是 attention 算子的输入。

不依赖 NPU backend，不计时，只验证语义。

In [ ]:
toy_q = torch.zeros(1, 1, 6, 1, dtype=torch.float32)
toy_k = torch.zeros_like(toy_q)
toy_v = torch.tensor([1, 2, 3, 10, 20, 30], dtype=torch.float32).view(1, 1, 6, 1)
toy_doc_ids = torch.tensor([0, 0, 0, 1, 1, 1])

toy_causal_out = F.scaled_dot_product_attention(toy_q, toy_k, toy_v, is_causal=True)
toy_positions = torch.arange(6)
toy_document_mask = (
    (toy_doc_ids[:, None] == toy_doc_ids[None, :])
    & (toy_positions[:, None] >= toy_positions[None, :])
)
toy_block_out = F.scaled_dot_product_attention(
    toy_q, toy_k, toy_v, attn_mask=toy_document_mask
)

labels_a = torch.tensor([-100, -100, 3, -100, -100, 30])
labels_b = torch.full_like(labels_a, -100)
label_only_out_a = F.scaled_dot_product_attention(toy_q, toy_k, toy_v, is_causal=True)
label_only_out_b = F.scaled_dot_product_attention(toy_q, toy_k, toy_v, is_causal=True)

print("first token of document 1, causal output:", float(toy_causal_out[0, 0, 3, 0]))
print("first token of document 1, block-document output:", float(toy_block_out[0, 0, 3, 0]))
print("changing labels changes attention:", not torch.equal(label_only_out_a, label_only_out_b))
print("labels are loss-only metadata:", labels_a.tolist(), "->", labels_b.tolist())


### 小结

这个 toy 实验用最少的 token 把三种 mask 的语义差别钉死了。

Q/K 全为零、V 为 `[1,2,3,10,20,30]`，前三个 token 属文档 0，后三个属文档 1。文档 1 的第一个 token（位置 3）在 causal mask 下输出 `4.0`——它平均读了 `[1,2,3,10]`，跨到了前一个文档。换成 document-aware mask 后输出变成 `10.0`——只能读自己文档内的 `[10]`。数值差了两倍多，不是 roundoff 级别的差异，而是彻底的语义改变。

然后保持 causal mask 不变，把 labels 从 `[-100,-100,3,-100,-100,30]` 全部改为 -100，attention 输出纹丝不动。这不是巧合——`F.scaled_dot_product_attention` 根本不接收 labels 参数，labels 只影响 loss 计算，进不了 attention 算子。

所以结论很明确：**packed SFT 只靠 `is_causal=True` 不够——它不管文档边界；`labels=IGNORE_INDEX` 也帮不上忙——它不进 attention。**正确的做法是把文档边界转成 attention mask（dense mask 或 VarLen 的 `cu_seq`），这正是 05.04 和 05.05 要解决的问题。

## 6. VarLen 语义与跳算收益

数值对照和性能 sweep 都固定 `T=4096`。先用 `4×1024` 的文档划分，将显式 block-document eager attention 作为语义 reference，与 TND `sparse_mode=7` 对比。

随后固定 `T=4096` 和完全相同的 Q/K/V shape，只改变累计长度：

| 划分 | 理论 pair 比例（相对单段 causal） | 理论跳过比例 |
|---|---:|---:|
| `1×4096` | ~100% | ~0% |
| `2×2048` | ~50% | ~50% |
| `4×1024` | ~25% | ~75% |
| `8×512` | ~12.5% | ~87.5% |

实测不会严格按理论比例缩放——tile、launch、带宽和固定开销仍然存在。实验只验证两件事：语义正确，且耗时随可计算 pair 减少呈下降趋势。如果不下降，说明当前 kernel/shape 没兑现理论潜力。

In [ ]:
check_lengths = [1024] * 4
check_cu_seq = make_cu_seq(check_lengths)
check_q, check_k, check_v = make_inputs()
check_allowed = document_allowed_mask(check_lengths, device=DEVICE)

with torch.no_grad():
    block_reference = eager_attention(check_q, check_k, check_v, allowed_mask=check_allowed)
    varlen_output = varlen_attention_v3(check_q, check_k, check_v, check_cu_seq)

varlen_diff = (block_reference - varlen_output).abs()
print("VarLen vs block-document eager max / mean abs:",
      float(varlen_diff.max()), float(varlen_diff.mean()))
del check_q, check_k, check_v, check_allowed, block_reference, varlen_output, varlen_diff

partitions = {
    "1x4096": [4096],
    "2x2048": [2048, 2048],
    "4x1024": [1024] * 4,
    "8x512": [512] * 8,
}

dense_fwd = benchmark_case(fusion_attention_v3)
dense_train = benchmark_case(fusion_attention_v3, backward=True)
dense_pairs = SEQ_LEN * (SEQ_LEN + 1) // 2

varlen_rows = []
for label, lengths in partitions.items():
    cu_seq = make_cu_seq(lengths)
    fn = lambda q, k, v, cu_seq=cu_seq: varlen_attention_v3(q, k, v, cu_seq)
    varlen_pairs = sum(length * (length + 1) // 2 for length in lengths)
    fwd_ms = benchmark_case(fn)
    train_ms = benchmark_case(fn, backward=True)
    varlen_rows.append((
        label,
        f"{varlen_pairs / dense_pairs:.3f}",
        f"{1 - varlen_pairs / dense_pairs:.1%}",
        f"{fwd_ms:.3f}",
        f"{dense_fwd / fwd_ms:.2f}x",
        f"{train_ms:.3f}",
        f"{dense_train / train_ms:.2f}x",
    ))

print("\ndense BSND sparse_mode=3 baseline:",
      f"forward={dense_fwd:.3f} ms, fwd+bwd={dense_train:.3f} ms")
print_rows(
    ("partition", "pair ratio", "theory skipped", "forward ms", "fwd speedup", "fwd+bwd ms", "train speedup"),
    varlen_rows,
)


### 小结

先从语义说起：VarLen（`sparse_mode=7`）与显式的 block-document eager attention 在 BF16 下最大绝对误差 `0.015625`，数值一致。这组对照确认了当前测试中被跳过的是 document mask 本来就禁止的跨文档 pair。

再看性能。固定 `T=4096`，只改变文档划分：

- `1×4096`（单文档 VarLen）：`1.073 ms`，比 dense BSND 的 `0.690 ms` 慢 55%；这个测量点没有跨文档 pair 可跳过。
- `2×2048`：`0.536 ms`，在这个测量点反超 dense，speedup 1.29×。
- `4×1024` 和 `8×512`：继续下降到 `0.311 ms` 和 `0.192 ms`，分别加速 2.22× 和 3.59×。

从这四个划分点可以观察到：在总 token 数相同、文档等长的对照里，文档增多、每段缩短时，理论 pair 数与实测耗时同时下降。不过 3.59× 仍明显低于理论 pair speedup 7.99×，说明 tile 粒度、kernel launch、HBM 带宽等成本没有随 pair 数线性缩小。§7 的 trace 中同时出现 `aclnnFlashAttentionVarLenScore`，device duration 从 dense 的 `2231.7 µs` 降到 `786.2 µs`，与 wall-time 的下降方向一致。

这些采样点支持两个具体判断：理论 pair 比例不能直接当作 wall-time 加速比；在本节配置中，dense 与 VarLen 的快慢关系在已测的 `1×4096` 和 `2×2048` 两点之间发生了反转。它们并不能确定一个适用于任意文档长度、shape 和软件栈的“≥2 个文档”临界点。

## 7. Trace：把 Wall-time 落到 Device Kernel

为 4 个 `B=1,S/T=4096` 代表路径各采集一次 forward+backward trace：

- eager / SDPA / dense V3：相同的单文档 causal 输入
- VarLen V3：相同 Q/K/V shape，仅把 `cu_seq` 划分为 `8×512`

报告带 `Task Type` 的 device kernel 数量、总 device duration 和最耗时 kernel。kernel 数用于验证融合程度；稳定性能仍以 §3 和 §6 的重复 wall-time 为准。

In [ ]:
TRACE_ROOT = Path("outputs/05.03_operator_validation")
TRACE_ROOT.mkdir(parents=True, exist_ok=True)


def capture_trace(name, fn):
    q, k, v = make_inputs(requires_grad=True)
    grad = torch.randn_like(q)
    trace_path = TRACE_ROOT / f"{name}.json"

    with torch_npu.profiler.profile(
        activities=[
            torch_npu.profiler.ProfilerActivity.CPU,
            torch_npu.profiler.ProfilerActivity.NPU,
        ],
        schedule=torch_npu.profiler.schedule(wait=0, warmup=1, active=1, repeat=1),
        record_shapes=True,
        profile_memory=True,
        experimental_config=torch_npu.profiler._ExperimentalConfig(
            profiler_level=torch_npu.profiler.ProfilerLevel.Level1,
        ),
    ) as prof:
        for _ in range(2):
            fn(q, k, v).backward(grad)
            torch.npu.synchronize()
            q.grad = k.grad = v.grad = None
            prof.step()

    prof.export_chrome_trace(str(trace_path))
    del q, k, v, grad
    return trace_path


def summarize_trace(trace_path):
    events = json.loads(trace_path.read_text(encoding="utf-8"))
    kernel_events = [
        event
        for event in events
        if str(event.get("args", {}).get("Task Type", "")).startswith(("MIX_", "AI_"))
    ]
    duration_by_name = Counter()
    for event in kernel_events:
        duration_by_name[event.get("name", "<unnamed>")] += float(event.get("dur", 0.0))
    return {
        "kernel_count": len(kernel_events),
        "device_us": sum(duration_by_name.values()),
        "top": duration_by_name.most_common(4),
    }


varlen_trace_cu = make_cu_seq([512] * 8)
trace_cases = {
    "eager_s4096": eager_attention,
    "sdpa_s4096": pytorch_sdpa,
    "fa_v3_s4096": fusion_attention_v3,
    "varlen_8x512_t4096": lambda q, k, v: varlen_attention_v3(
        q, k, v, varlen_trace_cu
    ),
}

trace_rows = []
for name, fn in trace_cases.items():
    trace_path = capture_trace(name, fn)
    summary = summarize_trace(trace_path)
    trace_rows.append((name, summary["kernel_count"], f"{summary['device_us']:.1f}", trace_path))
    print("\n", name, "top device kernels:")
    for kernel_name, duration_us in summary["top"]:
        print(f"  {duration_us:9.1f} us  {kernel_name}")

print("\ntrace summary:")
print_rows(("path", "device kernels", "device total us", "trace"), trace_rows)


### 小结

四条 trace 现在使用同一个 `S/T=4096`，可以直接比较而不受序列长度混杂影响。`eager_s4096` 应出现离散的矩阵乘、mask 和 softmax kernel；`sdpa_s4096` 必须出现 `FlashAttentionScore` 才能证明 dispatch 没有 fallback；`fa_v3_s4096` 用于观察直接调用原生融合算子时还剩多少 device kernel。

`varlen_8x512_t4096` 与 dense V3 的 Q/K/V shape 完全相同，唯一差别是累计长度。trace 中若出现 `aclnnFlashAttentionVarLenScore` / `aclnnFlashAttentionUnpaddingScoreGrad`，且 device duration 下降，就能把 §6 的 wall-time 趋势归因到 kernel 确实跳过了跨文档工作。

因此 trace 的判据是：**先看 kernel 名确认路径，再看 kernel 数确认融合程度，最后看 device duration 解释 wall-time；三者不能互相替代。**

## 8. 总结

本节所有 NPU 对照都固定 `S/T=4096`，因此同一实验内的路径可以直接比较。

**SDPA 是 dispatch 入口**：本节 trace 出现 `FlashAttentionScore`，才把对应 wall-time 归因给融合 backend。**FlashAttention 改变执行与 IO**：在 `[1,16,4096,4096]` 这一 shape 上，eager 的单个 BF16 score 就是 `512 MiB`；融合路径不物化它，但单一 shape 不能证明完整的渐近增长曲线。**GQA 缩小 K/V 侧**：`Nkv=16→8` 使 K/V 字节数减半、逻辑 score 不变；本组测量中 SDPA wall-time 只改善 6.3%/8.7%，不是随 K/V 字节数同比减半。**packed SFT 必须传 document-aware mask**：causal 只限制未来位置，`IGNORE_INDEX` 只影响 loss，二者都不能隔离文档。**VarLen 跳过跨文档计算**：本节四个 `T=4096` 划分点呈现 pair 数越少、耗时越低的趋势，但理论 pair speedup 不是 wall-time 保证，也不能从两个相邻测量点推出通用的文档数临界值。

依据这组实验，一个可检验的选择方式是：先用 SDPA，并用 trace 确认实际 backend；packed 多文档时同时验证 VarLen 的语义与性能；若输入接近单文档，则保留 dense BSND 对照，而不是预设任一路径必然更快。

> 换 NPU、驱动、CANN、PyTorch、`torch_npu` 或输入 shape 后，重新运行相应对照；上面的 wall-time 趋势不自动迁移。

## 练习

1. （单选题）`B=1,Nq=16,S=4096` 时，一个 BF16 逻辑 score `[B,Nq,S,S]` 的大小是多少？
    A. 32 MiB
    B. 128 MiB
    C. 512 MiB
    D. 1024 MiB

2. （判断题）只测 `S=4096` 的峰值显存，足以证明某条路径对所有 S 都按同一种渐近规律增长。

3. （判断题）VarLen 在单文档场景（1×4096）下必然比 dense BSND 更快。

4. （单选题）以下哪项是计时实验的正确做法？
    A. 不设 warmup，直接计时第一次调用
    B. 在计时区间前后加 `torch.npu.synchronize()`
    C. 用 profiler 内部的 kernel duration 替代重复 wall-time
    D. 只测一次，不取平均

5. （单选题）固定 `T=4096`，`8×512` 的理论 pair speedup 约为 7.99×。它与 wall-time 的关系是什么？
    A. wall-time 必然也加速 7.99×
    B. wall-time 可能低于 7.99×，因为还有 tile、launch、layout 与带宽开销
    C. wall-time 与 pair 数完全无关
    D. 只有 eager attention 才能获得该加速

In [ ]:
!cat ./answer/05.03_answer.txt
